# Ejercicio Módulo 2
**Inteligencia Artificial - CEIA - FIUBA**

**INSERTE AQUÍ SU NOMBRE**

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [1]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi

In [ ]:
def search_algorithm(number_disks=5) -> (NodeHanoi, dict):

    import heapq
    import itertools

    list_disks = [i for i in range(5, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)

    ##### EDITAR ESTA ZONA
    def heuristic(state: StatesHanoi) -> float:
        # Cantidad de discos que no estan en la 3er varilla
        if state == goal_state:
            return 0.0
        
        rods = state.get_state()
        return float(sum(len(rod) for rod in rods[:2]))

    # Métricas de rendimiento
    metrics = {
        "solution_found": False,
        "nodes_explored": 0,
        "states_visited": 0,
        "nodes_in_frontier": 0,
        "max_depth": 0,
        "cost_total": None,
    }

    # Inicializamos la frontera de A* con el nodo raíz
    start_node = NodeHanoi(initial_state)
    priority_queue = []
    counter = itertools.count()  # para desempatar en el heap

    g_start = getattr(start_node, "path_cost", 0.0)
    f_start = g_start + heuristic(initial_state)
    heapq.heappush(priority_queue, (f_start, next(counter), start_node))

    # Mejor costo g conocido para cada estado
    best_g = {initial_state: g_start}
    visited_states = set()

    solution = None

    while priority_queue:
        # Obtengo el proximo nodo a expandir
        f, _, node = heapq.heappop(priority_queue)
        metrics["nodes_explored"] += 1

        # Obtengo el estado del nodo (como esta cada torre) y lo guardo en el set de estados visitados
        state = node.state
        visited_states.add(state)
        metrics["states_visited"] = len(visited_states)

        #Calculo la profundidad maxima que se alcanzo
        depth = getattr(node, "depth", 0)
        if depth > metrics["max_depth"]:
            metrics["max_depth"] = depth

        # Si el estado es el objetivo, guardo el nodo como solucion y termino
        if problem.goal_test(state):
            solution = node
            metrics["solution_found"] = True
            metrics["cost_total"] = getattr(node, "path_cost", None)
            break

        # Como no llegamos al a solucion, expandimos sucesores
        for child in node.expand(problem):
            child_state = child.state
            g_child = getattr(child, "path_cost", best_g.get(state, 0.0) + 1.0)

            # Si el estado no esta en el diccionario de mejores costos o el costo del sucesor es menor que el costo almacenado, actualizo el diccionario y agrego el sucesor a la lista de nodos
            if child_state not in best_g or g_child < best_g[child_state]:
                best_g[child_state] = g_child
                f_child = g_child + heuristic(child_state)
                heapq.heappush(priority_queue, (f_child, next(counter), child))

    metrics["nodes_in_frontier"] = len(priority_queue)

    # Si no se encontró solución, devolvemos el nodo inicial como placeholder
    if solution is None:
        solution = start_node

    #####

    return solution, metrics

Se prueba la función:

In [21]:
solution, metrics = search_algorithm(number_disks=5)

Veamos las métricas:

In [22]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 178
states_visited: 178
nodes_in_frontier: 15
max_depth: 31
cost_total: 31.0


Veamos el camino de estados desde el principio a la solución:

In [5]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 | 4 | 3 2 1>
<Node HanoiState: 5 | 4 1 | 3 2>
<Node HanoiState: 5 2 | 4 1 | 3>
<Node HanoiState: 5 2 1 | 4 | 3>
<Node HanoiState: 5 2 1 | 4 3 | >
<Node HanoiState: 5 2 | 4 3 | 1>
<Node HanoiState: 5 | 4 3 2 | 1>
<Node HanoiState: 5 | 4 3 2 1 | >
<Node HanoiState:  | 4 3 2 1 | 5>
<Node HanoiState: 1 | 4 3 2 | 5>
<Node HanoiState: 1 | 4 3 | 5 2>
<Node HanoiState:  | 4 3 | 5 2 1>
<Node HanoiState: 3 | 4 | 5 2 1>
<Node HanoiState: 3 | 4 1 | 5 2>
<Node HanoiState: 3 2 | 4 1 | 5>
<Node HanoiState: 3 2 1 | 4 | 5>
<Node HanoiState: 3 2 1 |  | 5 4>
<Node HanoiState: 3 2 |  | 5 4 1>
<Node HanoiState: 3 | 2 | 5 4 1>
<Node HanoiState: 3 | 2 1 | 5 4>
<Node HanoiState:  | 2 1 | 5 4 3>
<Node HanoiState: 1 | 2 | 5 4 

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [6]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 4 from 1 to 2
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 5 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 3 from 2 to 1
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 4 from 2 to 3
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
